# 01 — Light-curve sanity check

Run this only in a Lightkurve-compatible environment. It creates a small, stratified SPOC 120-second pilot before any full-catalogue download.

In [ ]:
from pathlib import Path
import pandas as pd
from transit_hunter.coverage import build_coverage_manifest

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
labels = pd.read_csv(ROOT / 'data/metadata/labels_snapshot.csv')
pilot = (labels.groupby('tfopwg_disp', group_keys=False).apply(lambda group: group.sample(min(10, len(group)), random_state=4000), include_groups=False).reset_index())
pilot[['toi', 'tid', 'tfopwg_disp', 'label']]

In [ ]:
coverage = build_coverage_manifest(pilot)  # metadata query only; no FITS download
coverage.to_csv(ROOT / 'data/metadata/pilot_coverage_manifest.csv', index=False)
coverage['coverage_status'].value_counts()

In [ ]:
from transit_hunter.download import load_tic_arrays

available = coverage.loc[coverage.coverage_status.eq('available')].iloc[0]
time, flux = load_tic_arrays(int(available.tid), ROOT / 'data/raw')
import matplotlib.pyplot as plt
plt.plot(time, flux, '.', ms=1)
plt.title(f"TIC {int(available.tid)} — raw SPOC 120-second light curve")
plt.xlabel('Time [BJD - 2457000]'); plt.ylabel('Flux')